# 01. DENUE Data Ingest and Geospatial Filtering

Validates downloading, decoding, radius filtering (vectorized Haversine), strata filtering, and parquet persistence.

**Philosophy:** Each code cell defines a self-contained, typed function annotated with `# MIGRATES TO:` for future extraction into `src/mx_osint/sources/denue_loader.py`.

In [ ]:
# MIGRATES TO: src/mx_osint/config.py::Settings
from pathlib import Path
from pydantic_settings import BaseSettings, SettingsConfigDict
from pydantic import Field

class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env", env_file_encoding="utf-8", extra="ignore")
    INEGI_STATE_CODE: str = Field(default="11", description="INEGI State Code (01-32)")
    TARGET_LAT: float = Field(default=21.1221, description="Latitude center")
    TARGET_LON: float = Field(default=-101.6826, description="Longitude center")
    RADIUS_KM: float = Field(default=3.0, description="Catchment radius in km")
    MIN_EMPLOYEES_STRATA: str = Field(default="11 a 30 personas", description="Minimum employee strata")
    DDG_SLEEP_SEC: float = Field(default=2.0, description="DDG rate limit sleep")
    CACHE_DIR: Path = Field(default=Path("data/raw"), description="Local cache directory")

settings = Settings()
print("Config loaded successfully:", settings.model_dump())

In [ ]:
# MIGRATES TO: src/mx_osint/sources/denue_loader.py::download_denue_state
import hashlib
from pathlib import Path
import requests
from tenacity import retry, stop_after_attempt, wait_exponential
from loguru import logger

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def download_denue_state(state_code: str, cache_dir: Path) -> Path:
    """Downloads DENUE state zip package from official INEGI portal with retry and caching."""
    cache_dir.mkdir(parents=True, exist_ok=True)
    target_file = cache_dir / f"denue_{state_code}_csv.zip"
    if target_file.exists() and target_file.stat().st_size > 0:
        logger.info(f"Cache hit for state {state_code}: {target_file}")
        return target_file
    url = f"https://www.inegi.org.mx/contenidos/masiva/denue/denue_{state_code}_csv.zip"
    logger.info(f"Downloading DENUE state {state_code} from {url}...")
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()
    hasher = hashlib.sha256()
    with open(target_file, "wb") as f:
        for chunk in response.iter_content(chunk_size=65536):
            if chunk:
                f.write(chunk)
                hasher.update(chunk)
    logger.info(f"Saved {target_file} (SHA256: {hasher.hexdigest()[:12]})")
    return target_file

In [ ]:
# MIGRATES TO: src/mx_osint/sources/denue_loader.py::read_denue_csv
import zipfile
from pathlib import Path
import polars as pl
from loguru import logger

def read_denue_csv(zip_or_csv_path: Path) -> pl.DataFrame:
    """Reads raw DENUE data using latin1/utf8-lossy encoding and selects standardized columns."""
    cols_needed = [
        "id", "clee", "nombre_estab", "raz_social", "codigo_act", "nombre_act",
        "per_ocu", "telefono", "correoelec", "www", "lat", "lon", "ent", "mun", "fecha_alta"
    ]
    if zip_or_csv_path.suffix.lower() == ".parquet":
        df = pl.read_parquet(zip_or_csv_path)
        return df.select([c for c in cols_needed if c in df.columns])
    
    # If zip, locate inner csv
    if zip_or_csv_path.suffix.lower() == ".zip":
        with zipfile.ZipFile(zip_or_csv_path, 'r') as z:
            csv_names = [n for n in z.namelist() if n.lower().endswith('.csv') and not n.startswith('__MACOSX')]
            if not csv_names:
                raise FileNotFoundError(f"No CSV found inside {zip_or_csv_path}")
            with z.open(csv_names[0]) as f:
                df = pl.read_csv(
                    f.read(),
                    encoding="utf8-lossy",
                    ignore_errors=True,
                    infer_schema_length=1000
                )
    else:
        df = pl.read_csv(
            zip_or_csv_path,
            encoding="utf8-lossy",
            ignore_errors=True,
            infer_schema_length=1000
        )
    
    # Standardize column names to lowercase
    df = df.rename({c: c.lower() for c in df.columns})
    present_cols = [c for c in cols_needed if c in df.columns]
    logger.info(f"Read {len(df)} rows with {len(present_cols)} standardized columns.")
    return df.select(present_cols)

In [ ]:
# MIGRATES TO: src/mx_osint/sources/denue_loader.py::filter_by_radius
import math
import polars as pl

def filter_by_radius(df: pl.DataFrame, target_lat: float, target_lon: float, radius_km: float) -> pl.DataFrame:
    """Vectorized Haversine distance calculation and radius filtering using Polars expression API."""
    r_km = 6371.0
    rad = math.pi / 180.0
    t_lat_rad = target_lat * rad
    
    lat_rad = pl.col("lat") * rad
    dlat = (pl.col("lat") - target_lat) * rad
    dlon = (pl.col("lon") - target_lon) * rad
    
    a = (
        (dlat / 2.0).sin().pow(2)
        + math.cos(t_lat_rad) * lat_rad.cos() * (dlon / 2.0).sin().pow(2)
    )
    c = 2.0 * a.sqrt().arcsin()
    dist_expr = (c * r_km).alias("distance_km")
    
    return (
        df.with_columns(dist_expr)
        .filter(pl.col("distance_km") <= radius_km)
        .sort("distance_km")
    )

In [ ]:
# MIGRATES TO: src/mx_osint/sources/denue_loader.py::filter_by_strata
import polars as pl

STRATA_ORDER = [
    "0 a 5 personas",
    "6 a 10 personas",
    "11 a 30 personas",
    "31 a 50 personas",
    "51 a 100 personas",
    "101 a 250 personas",
    "251 y más personas"
]

def filter_by_strata(df: pl.DataFrame, min_strata: str) -> pl.DataFrame:
    """Filters records by employee strata threshold or higher."""
    if min_strata not in STRATA_ORDER:
        raise ValueError(f"Invalid strata '{min_strata}'. Must be one of {STRATA_ORDER}")
    allowed = STRATA_ORDER[STRATA_ORDER.index(min_strata):]
    return df.filter(pl.col("per_ocu").is_in(allowed))

In [ ]:
# MIGRATES TO: src/mx_osint/sources/denue_loader.py::persist_raw
from pathlib import Path
import polars as pl
from loguru import logger

def persist_raw(df: pl.DataFrame, state_code: str, radius_km: float, output_dir: Path = Path("data/raw")) -> Path:
    """Persists filtered DENUE extract to parquet cache and logs summary statistics."""
    output_dir.mkdir(parents=True, exist_ok=True)
    dest = output_dir / f"denue_state_{state_code}_radius_{int(radius_km)}km.parquet"
    df.write_parquet(dest)
    logger.info(f"Persisted {len(df)} records to {dest}")
    
    # Summary statistics
    total = len(df)
    if total > 0:
        has_email_pct = (df.filter(pl.col("correoelec").is_not_null() & (pl.col("correoelec") != "")).height / total) * 100
        has_www_pct = (df.filter(pl.col("www").is_not_null() & (pl.col("www") != "")).height / total) * 100
        has_tel_pct = (df.filter(pl.col("telefono").is_not_null() & (pl.col("telefono") != "")).height / total) * 100
        print(f"\n--- Ingest Summary ({total} records within {radius_km}km) ---")
        print(f"Email coverage: {has_email_pct:.1f}%")
        print(f"Website coverage: {has_www_pct:.1f}%")
        print(f"Phone coverage: {has_tel_pct:.1f}%")
    return dest

In [ ]:
# Execution and validation harness
from pathlib import Path
import polars as pl

print("Running ingest validation pipeline...")
# Attempt real download or fallback to synthetic fixture
fixture_file = Path("fixtures/synthetic_denue_sample.parquet")
try:
    raw_source = download_denue_state(settings.INEGI_STATE_CODE, settings.CACHE_DIR)
    raw_df = read_denue_csv(raw_source)
except Exception as e:
    print(f"Network download skipped/failed ({e}). Falling back to fixture: {fixture_file}")
    raw_df = pl.read_parquet(fixture_file)

# Apply radius filter
geo_df = filter_by_radius(raw_df, settings.TARGET_LAT, settings.TARGET_LON, settings.RADIUS_KM)
print(f"Records within {settings.RADIUS_KM}km: {len(geo_df)}")

# Apply strata filter
strata_df = filter_by_strata(geo_df, settings.MIN_EMPLOYEES_STRATA)
print(f"Records >= '{settings.MIN_EMPLOYEES_STRATA}': {len(strata_df)}")

# Persist extract
persist_path = persist_raw(strata_df, settings.INEGI_STATE_CODE, settings.RADIUS_KM)
print(f"Validation complete. Output cached at {persist_path}")